In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================
# 0. Метрика
# ============================================================
def wape_plus_rbias(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    s = y_true.sum()
    if s == 0:
        return 0.0
    return np.abs(y_pred - y_true).sum() / s + np.abs(y_pred.sum() / s - 1)

In [3]:
# ============================================================
# 1. Загрузка
# ============================================================
train = pd.read_parquet('train_team_track.parquet')
test = pd.read_parquet('test_team_track.parquet')
train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp'] = pd.to_datetime(test['timestamp'])

route_to_office = train.groupby('route_id')['office_from_id'].first().to_dict()
test['office_from_id'] = test['route_id'].map(route_to_office)

status_cols = [f'status_{i}' for i in range(1, 9)]
for c in status_cols:
    if c not in test.columns:
        test[c] = np.nan

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

test_steps_per_route = test.groupby('route_id').size()
print(f"Test steps per route: min={test_steps_per_route.min()}, "
      f"max={test_steps_per_route.max()}, median={test_steps_per_route.median()}")
print(f"Total test timestamps: {test['timestamp'].nunique()}")

Device: cuda
Test steps per route: min=10, max=10, median=10.0
Total test timestamps: 10


In [4]:
# ============================================================
# 2. Нормализация: Log1p + RobustScaler
# ============================================================
class LogRobustScaler:
    def __init__(self):
        self.medians = {}
        self.iqrs = {}
        self.use_log = {}

    def fit(self, df, columns, log_columns=None):
        log_columns = log_columns or []
        for col in columns:
            v = df[col].dropna().values.copy()
            if col in log_columns:
                v = np.log1p(np.clip(v, 0, None))
                self.use_log[col] = True
            else:
                self.use_log[col] = False
            q25, q75 = np.percentile(v, [25, 75])
            self.medians[col] = np.median(v)
            self.iqrs[col] = max(q75 - q25, 1e-5)
        return self

    def transform(self, values, col):
        v = np.array(values, dtype=float)
        if self.use_log.get(col, False):
            v = np.log1p(np.clip(v, 0, None))
        return (v - self.medians[col]) / self.iqrs[col]

    def inverse_transform(self, values, col):
        v = np.array(values, dtype=float)
        v = v * self.iqrs[col] + self.medians[col]
        if self.use_log.get(col, False):
            v = np.expm1(v)
        return v

log_cols = status_cols + ['target_2h']
scaler = LogRobustScaler()
scaler.fit(train, log_cols, log_columns=log_cols)

In [5]:
# ============================================================
# 3. Временные признаки
# ============================================================
def add_time_features(df):
    df = df.copy()
    df['time_slot'] = df['timestamp'].dt.hour * 2 + df['timestamp'].dt.minute // 30
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour'] = df['timestamp'].dt.hour
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 48)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 48)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return df

train = add_time_features(train)
test = add_time_features(test)

time_features = ['hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
                 'dow_sin', 'dow_cos', 'is_weekend']

seq_cols = status_cols + ['target_2h']  # 9 нормализуемых
num_seq_features = len(seq_cols) + len(time_features)  # 9 + 7 = 16

In [7]:
# ============================================================
# 4. Построение окон
# ============================================================
WINDOW_SIZE = 48  # 24 часа — достаточно для суточного цикла

print(f"\nWindow: {WINDOW_SIZE} steps, Test horizon: 10 steps")
print(f"At test step 9: {WINDOW_SIZE - 9}/{WINDOW_SIZE} = "
      f"{(WINDOW_SIZE - 9) / WINDOW_SIZE * 100:.0f}% real data in window")

train_sorted = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

print("Building sequences...")
all_X, all_y, all_rids, all_oids, all_ts = [], [], [], [], []

for rid in sorted(train_sorted['route_id'].unique()):
    rd = train_sorted[train_sorted['route_id'] == rid]
    if len(rd) <= WINDOW_SIZE:
        continue

    norm_parts = [scaler.transform(rd[c].values, c) for c in seq_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    full_arr = np.column_stack([norm_arr, time_arr]).astype(np.float32)
    targets_norm = scaler.transform(rd['target_2h'].values, 'target_2h')
    oid = rd['office_from_id'].iloc[0]

    for i in range(WINDOW_SIZE, len(rd)):
        all_X.append(full_arr[i - WINDOW_SIZE:i])
        all_y.append(targets_norm[i])
        all_rids.append(rid)
        all_oids.append(oid)
        all_ts.append(rd['timestamp'].iloc[i])

all_X = np.nan_to_num(np.array(all_X, dtype=np.float32), nan=0.0)
all_y = np.array(all_y, dtype=np.float32)
all_rids = np.array(all_rids, dtype=np.int64)
all_oids = np.array(all_oids, dtype=np.int64)
all_ts = np.array(all_ts)

print(f"Sequences: {all_X.shape}")


Window: 48 steps, Test horizon: 10 steps
At test step 9: 39/48 = 81% real data in window
Building sequences...
Sequences: (4294000, 48, 16)


In [8]:
# ============================================================
# 5. Валидация: имитируем ТОЧНО тестовую ситуацию
#    Берём последние 10 шагов каждого маршрута как val
# ============================================================
# Это точно имитирует тест: 10 шагов после длинной истории

# Находим для каждого маршрута последние 10 временных точек
last_10_timestamps = sorted(train['timestamp'].unique())[-10:]
print(f"\nValidation timestamps (last 10): {last_10_timestamps[0]} — {last_10_timestamps[-1]}")

val_mask = np.isin(all_ts, last_10_timestamps)
train_mask = ~val_mask

X_tr, y_tr = all_X[train_mask], all_y[train_mask]
r_tr, o_tr = all_rids[train_mask], all_oids[train_mask]
X_va, y_va = all_X[val_mask], all_y[val_mask]
r_va, o_va = all_rids[val_mask], all_oids[val_mask]

print(f"Train: {X_tr.shape[0]:,}, Val: {X_va.shape[0]:,}")


Validation timestamps (last 10): 2025-05-30 06:00:00 — 2025-05-30 10:30:00
Train: 4,284,000, Val: 10,000


In [9]:
# ============================================================
# 6. Dataset / DataLoader
# ============================================================
class TSDataset(Dataset):
    def __init__(self, X, y, r, o):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.r = torch.LongTensor(r)
        self.o = torch.LongTensor(o)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.r[i], self.o[i], self.y[i]

BS = 1024
train_loader = DataLoader(TSDataset(X_tr, y_tr, r_tr, o_tr),
                          batch_size=BS, shuffle=True, pin_memory=True)
val_loader = DataLoader(TSDataset(X_va, y_va, r_va, o_va),
                        batch_size=BS * 2, shuffle=False, pin_memory=True)

In [7]:
# ============================================================
# 7. Модель
# ============================================================
class DeliveryNet(nn.Module):
    def __init__(self, nf=16, nr=1000, no=54,
                 re=32, oe=16, ch=64, lh=128, ll=2, dr=0.2):
        super().__init__()
        self.r_emb = nn.Embedding(nr, re)
        self.o_emb = nn.Embedding(no, oe)

        self.cnn = nn.Sequential(
            nn.Conv1d(nf, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
            nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
        )
        self.lstm = nn.LSTM(ch, lh, ll, batch_first=True,
                            dropout=dr if ll > 1 else 0)
        self.ln = nn.LayerNorm(lh)

        # Два выхода: attention pooling + последний hidden state
        self.attn = nn.Sequential(
            nn.Linear(lh, lh // 2), nn.Tanh(), nn.Linear(lh // 2, 1))

        # Используем и attention context, и last hidden
        head_in = lh * 2 + re + oe
        self.head = nn.Sequential(
            nn.Linear(head_in, 128), nn.GELU(), nn.Dropout(dr),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dr),
            nn.Linear(64, 1)
        )

    def forward(self, x, rid, oid):
        h = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)
        h, _ = self.lstm(h)
        h = self.ln(h)

        # Attention context
        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h * w).sum(dim=1)

        # Last hidden state (самый свежий)
        last = h[:, -1, :]

        r = self.r_emb(rid)
        o = self.o_emb(oid)
        return self.head(torch.cat([ctx, last, r, o], dim=1)).squeeze(-1)

In [ ]:
model = DeliveryNet(nf=num_seq_features).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [12]:
# ============================================================
# 8. Обучение
# ============================================================
class WapeBiasLoss(nn.Module):
    def __init__(self, bias_w=0.3):
        super().__init__()
        self.bias_w = bias_w
    def forward(self, p, t):
        return (p - t).abs().mean() + self.bias_w * (p.mean() - t.mean()).abs()

criterion = WapeBiasLoss(0.3)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-5)

EPOCHS = 2
PATIENCE = 12
best_metric, best_epoch, best_state = float('inf'), 0, None

print("\nTraining...")
for ep in range(EPOCHS):
    model.train()
    losses = []
    for xb, rb, ob, yb in train_loader:
        xb, rb, ob, yb = [t.to(device) for t in [xb, rb, ob, yb]]
        optimizer.zero_grad()
        loss = criterion(model(xb, rb, ob), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for xb, rb, ob, yb in val_loader:
            vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
            vt.append(yb.numpy())

    vp_d = np.clip(scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
    vt_d = scaler.inverse_transform(np.concatenate(vt), 'target_2h')
    m = wape_plus_rbias(vt_d, vp_d)

    if m < best_metric:
        best_metric, best_epoch = m, ep
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if ep % 3 == 0 or m <= best_metric:
        w = np.abs(vp_d - vt_d).sum() / vt_d.sum()
        b = np.abs(vp_d.sum() / vt_d.sum() - 1)
        print(f"Ep {ep:3d} | L {np.mean(losses):.4f} | "
              f"M {m:.4f} (W:{w:.4f} B:{b:.4f})"
              f"{' *' if m <= best_metric else ''}")

    if ep - best_epoch >= PATIENCE:
        print(f"Early stop at {ep}")
        break

print(f"\nBest: ep {best_epoch}, metric {best_metric:.6f}")
model.load_state_dict(best_state)
model = model.to(device)


Training...
Ep   0 | L 0.1306 | M 0.1612 (W:0.1458 B:0.0155) *

Best: ep 0, metric 0.161215


In [13]:
model.eval()
vp, vt = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = scaler.inverse_transform(np.concatenate(vt), 'target_2h')
m = wape_plus_rbias(vt_d, vp_d)
w = np.abs(vp_d - vt_d).sum() / vt_d.sum()
b = np.abs(vp_d.sum() / vt_d.sum() - 1)
print(f"Ep {ep:3d} | L {np.mean(losses):.4f} | "
      f"M {m:.4f} (W:{w:.4f} B:{b:.4f})"
      f"{' *' if m <= best_metric else ''}")

Ep   1 | L 0.1297 | M 0.1612 (W:0.1458 B:0.0155) *


In [8]:
time_features = ['hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
                 'dow_sin', 'dow_cos', 'is_weekend']

seq_cols = status_cols + ['target_2h']  # 9 нормализуемых
num_seq_features = len(seq_cols) + len(time_features)  # 9 + 7 = 16
model = DeliveryNet(nf=num_seq_features).to(device)
model.load_state_dict(torch.load('model_weights.pth'))

<All keys matched successfully>

In [12]:
train_sorted = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
WINDOW_SIZE = 48

In [13]:
# ============================================================
# 9. Рекурсивное предсказание (всего 10 шагов!)
# ============================================================
print("\nTest prediction (10 steps, mostly real data)...")

# Полная история для каждого маршрута
route_histories = {}
route_raw_statuses = {}

for rid in sorted(train_sorted['route_id'].unique()):
    rd = train_sorted[train_sorted['route_id'] == rid].sort_values('timestamp')
    norm_parts = [scaler.transform(rd[c].values, c) for c in seq_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    full = np.column_stack([norm_arr, time_arr]).astype(np.float32)
    route_histories[rid] = list(full)
    route_raw_statuses[rid] = rd[status_cols].values.tolist()

# Сортируем для рекурсии, но НЕ сбрасываем индекс
test_sorted = test.sort_values(['timestamp', 'route_id'])
test_timestamps = sorted(test['timestamp'].unique())

# Словарь: оригинальный id → предсказание
id_to_pred = {}

model.eval()

for step, ts in enumerate(test_timestamps):
    real_pct = (WINDOW_SIZE - step) / WINDOW_SIZE * 100
    print(f"  Step {step+1}/10: {ts} | {real_pct:.0f}% real statuses in window")

    batch_df = test_sorted[test_sorted['timestamp'] == ts]
    rids = batch_df['route_id'].values.astype(int)
    oids = batch_df['office_from_id'].values.astype(int)
    # Сохраняем оригинальные id из этого батча
    batch_ids = batch_df['id'].values

    # Собираем окна
    batch_X = []
    for rid in rids:
        h = route_histories[rid]
        if len(h) >= WINDOW_SIZE:
            w = np.array(h[-WINDOW_SIZE:], dtype=np.float32)
        else:
            pad = WINDOW_SIZE - len(h)
            w = np.array([h[0]] * pad + list(h), dtype=np.float32)
        batch_X.append(w)

    batch_X = np.nan_to_num(np.array(batch_X), nan=0.0)

    with torch.no_grad():
        preds_norm = model(
            torch.FloatTensor(batch_X).to(device),
            torch.LongTensor(rids).to(device),
            torch.LongTensor(oids).to(device)
        ).cpu().numpy()

    preds = np.clip(scaler.inverse_transform(preds_norm, 'target_2h'), 0, None)

    # Обновляем историю и сохраняем предсказания по id
    for j, (_, row) in enumerate(batch_df.iterrows()):
        rid = int(row['route_id'])
        original_id = row['id']

        # Привязываем предсказание к оригинальному id
        id_to_pred[original_id] = preds[j]

        # Статусы: скользящее среднее последних 4 реальных/приближённых
        raw = route_raw_statuses[rid]
        if len(raw) >= 4:
            approx = np.mean(raw[-4:], axis=0)
        else:
            approx = np.array(raw[-1]) if raw else np.zeros(8)

        norm_s = [scaler.transform(np.array([approx[k]]), status_cols[k])[0]
                  for k in range(8)]
        t_norm = preds_norm[j]
        t_vals = [row[f] for f in time_features]

        new_step = np.array(norm_s + [t_norm] + t_vals, dtype=np.float32)
        route_histories[rid].append(new_step)
        route_raw_statuses[rid].append(approx.tolist())




Test prediction (10 steps, mostly real data)...
  Step 1/10: 2025-05-30 11:00:00 | 100% real statuses in window
  Step 2/10: 2025-05-30 11:30:00 | 98% real statuses in window
  Step 3/10: 2025-05-30 12:00:00 | 96% real statuses in window
  Step 4/10: 2025-05-30 12:30:00 | 94% real statuses in window
  Step 5/10: 2025-05-30 13:00:00 | 92% real statuses in window
  Step 6/10: 2025-05-30 13:30:00 | 90% real statuses in window
  Step 7/10: 2025-05-30 14:00:00 | 88% real statuses in window
  Step 8/10: 2025-05-30 14:30:00 | 85% real statuses in window
  Step 9/10: 2025-05-30 15:00:00 | 83% real statuses in window
  Step 10/10: 2025-05-30 15:30:00 | 81% real statuses in window


In [14]:
# ============================================================
# 10. Сохранение — привязка через id
# ============================================================
submission = pd.DataFrame({
    'id': list(id_to_pred.keys()),
    'y_pred': list(id_to_pred.values())
})
submission = submission.sort_values('id').reset_index(drop=True)

# Проверки
assert len(submission) == len(test), \
    f"Row count mismatch: {len(submission)} vs {len(test)}"
assert set(submission['id']) == set(test['id']), "Missing or extra ids!"
assert submission['id'].nunique() == len(submission), "Duplicate ids!"
assert submission['y_pred'].isna().sum() == 0, "NaN in predictions!"
assert (submission['y_pred'] >= 0).all(), "Negative predictions!"

submission.to_csv('submission_nn.csv', index=False)
print(f"\nDone! {submission.shape}")
print(f"Sum: {submission['y_pred'].sum():.0f}")
print(submission.head(10))


Done! (10000, 2)
Sum: 614129
   id     y_pred
0   0  87.290255
1   1  89.625691
2   2  79.205621
3   3  16.638168
4   4  12.271226
5   5   9.864187
6   6   9.151147
7   7   8.454205
8   8   8.132324
9   9   8.451672


In [14]:
# ============================================================
# 9. Рекурсивное предсказание (всего 10 шагов!)
# ============================================================
print("\nTest prediction (10 steps, mostly real data)...")

# Полная история для каждого маршрута
route_histories = {}
route_raw_statuses = {}

for rid in sorted(train_sorted['route_id'].unique()):
    rd = train_sorted[train_sorted['route_id'] == rid].sort_values('timestamp')
    norm_parts = [scaler.transform(rd[c].values, c) for c in seq_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    full = np.column_stack([norm_arr, time_arr]).astype(np.float32)
    route_histories[rid] = list(full)
    route_raw_statuses[rid] = rd[status_cols].values.tolist()

test_sorted = test.sort_values(['timestamp', 'route_id'])
test_timestamps = sorted(test['timestamp'].unique())
predictions = {}

model.eval()

for step, ts in enumerate(test_timestamps):
    real_pct = (WINDOW_SIZE - step) / WINDOW_SIZE * 100
    print(f"  Step {step+1}/10: {ts} | {real_pct:.0f}% real statuses in window")

    batch_df = test_sorted[test_sorted['timestamp'] == ts]
    rids = batch_df['route_id'].values.astype(int)
    oids = batch_df['office_from_id'].values.astype(int)

    # Собираем окна
    batch_X = []
    for rid in rids:
        h = route_histories[rid]
        if len(h) >= WINDOW_SIZE:
            w = np.array(h[-WINDOW_SIZE:], dtype=np.float32)
        else:
            pad = WINDOW_SIZE - len(h)
            w = np.array([h[0]] * pad + list(h), dtype=np.float32)
        batch_X.append(w)

    batch_X = np.nan_to_num(np.array(batch_X), nan=0.0)

    with torch.no_grad():
        preds_norm = model(
            torch.FloatTensor(batch_X).to(device),
            torch.LongTensor(rids).to(device),
            torch.LongTensor(oids).to(device)
        ).cpu().numpy()

    preds = np.clip(scaler.inverse_transform(preds_norm, 'target_2h'), 0, None)

    # Обновляем историю
    for j, (_, row) in enumerate(batch_df.iterrows()):
        rid = int(row['route_id'])

        # Статусы: скользящее среднее последних 4 реальных/приближённых
        raw = route_raw_statuses[rid]
        if len(raw) >= 4:
            approx = np.mean(raw[-4:], axis=0)
        else:
            approx = np.array(raw[-1]) if raw else np.zeros(8)

        norm_s = [scaler.transform(np.array([approx[k]]), status_cols[k])[0]
                  for k in range(8)]
        t_norm = preds_norm[j]
        t_vals = [row[f] for f in time_features]

        new_step = np.array(norm_s + [t_norm] + t_vals, dtype=np.float32)
        route_histories[rid].append(new_step)
        route_raw_statuses[rid].append(approx.tolist())

        predictions[(rid, ts)] = preds[j]


Test prediction (10 steps, mostly real data)...
  Step 1/10: 2025-05-30 11:00:00 | 100% real statuses in window
  Step 2/10: 2025-05-30 11:30:00 | 98% real statuses in window
  Step 3/10: 2025-05-30 12:00:00 | 96% real statuses in window
  Step 4/10: 2025-05-30 12:30:00 | 94% real statuses in window
  Step 5/10: 2025-05-30 13:00:00 | 92% real statuses in window
  Step 6/10: 2025-05-30 13:30:00 | 90% real statuses in window
  Step 7/10: 2025-05-30 14:00:00 | 88% real statuses in window
  Step 8/10: 2025-05-30 14:30:00 | 85% real statuses in window
  Step 9/10: 2025-05-30 15:00:00 | 83% real statuses in window
  Step 10/10: 2025-05-30 15:30:00 | 81% real statuses in window


In [15]:
# ============================================================
# 10. Калибровка и сохранение
# ============================================================
# Bias на валидации
model.eval()
vp, vt = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = scaler.inverse_transform(np.concatenate(vt), 'target_2h')
bias_ratio = vt_d.sum() / vp_d.sum()

print(f"\nVal metric: {wape_plus_rbias(vt_d, vp_d):.6f}")
print(f"Bias ratio: {bias_ratio:.4f}")


Val metric: 0.161215
Bias ratio: 1.0157


In [16]:
test['target_2h'] = test.apply(
    lambda r: predictions.get((r['route_id'], r['timestamp']), 0), axis=1)

if abs(bias_ratio - 1.0) > 0.005:
    test['target_2h'] = np.clip(test['target_2h'] * bias_ratio, 0, None)
    print(f"Applied calibration: ×{bias_ratio:.4f}")

sub = test[['route_id', 'timestamp', 'target_2h']].sort_values(
    ['route_id', 'timestamp']).reset_index(drop=True)
sub.to_csv('submission_nn.csv', index=False)

print(f"\nDone! {sub.shape}")
print(f"Sum: {sub['target_2h'].sum():.0f}")
print(sub.head(10))

Applied calibration: ×1.0157

Done! (10000, 3)
Sum: 623767
   route_id           timestamp  target_2h
0         0 2025-05-30 11:00:00   9.815778
1         0 2025-05-30 11:30:00  11.221309
2         0 2025-05-30 12:00:00   3.047487
3         0 2025-05-30 12:30:00   3.586887
4         0 2025-05-30 13:00:00   3.836302
5         0 2025-05-30 13:30:00   2.556772
6         0 2025-05-30 14:00:00   2.633702
7         0 2025-05-30 14:30:00   2.560078
8         0 2025-05-30 15:00:00   2.569975
9         0 2025-05-30 15:30:00   2.276032


In [19]:
res = test[['id', 'target_2h']]

In [30]:
res.to_csv('fst.csv', index=False)

In [28]:
res = res.rename(columns={'target_2h': 'y_pred'})

In [29]:
res

,id,y_pred
0,0,88.660277
1,1,91.032367
2,2,80.448753
3,3,16.899304
4,4,12.463823
...,...,...
9995,9995,86.848996
9996,9996,87.707737
9997,9997,87.931995
9998,9998,89.106573


In [31]:
torch.save(model.state_dict(), 'model_weights.pth')